# Calculate silica group index and dispersion in Python

Compare phase index with group index and calculate the group-delay dispersion
(GDD) through 1 mm of fused silica. The plots below show why the index used for
phase propagation differs from the index used for pulse arrival time.

Run all cells in order. The first cell installs PyOptik.


In [ ]:
%pip install "PyOptik>=3.3.0"


## Calculate and plot

The first run downloads the RefractiveIndex.INFO snapshot and requires internet access; later runs reuse the local cache.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from TypedUnit import ureg
from PyOptik import material

# First run downloads the material snapshot; subsequent runs reuse it.
silica = material("fused silica")  # Documented default: main/SiO2/Malitson
wavelengths = np.linspace(500, 1600, 300) * ureg.nanometer
length = 1 * ureg.millimeter

phase_index = silica.n(wavelengths, out_of_range="raise")
group_index = silica.compute_group_index(wavelengths)
gdd = silica.compute_group_delay_dispersion(wavelengths, length=length)

figure, axes = plt.subplots(1, 2, figsize=(10, 4), layout="constrained")
axes[0].plot(wavelengths.magnitude, phase_index, label="Phase index n")
axes[0].plot(wavelengths.magnitude, group_index, label="Group index n_g")
axes[0].set(xlabel="Vacuum wavelength [nm]", ylabel="Refractive index",
            title="Fused silica: phase and group index")
axes[0].legend()
axes[1].plot(wavelengths.magnitude, gdd.to(ureg.femtosecond**2).magnitude)
axes[1].axhline(0, color="black", linewidth=0.8)
axes[1].set(xlabel="Vacuum wavelength [nm]", ylabel="GDD [fs²]",
            title="Dispersion through 1 mm of fused silica")
for axis in axes:
    axis.grid(alpha=0.25)

sample = 800 * ureg.nanometer
print(f"Group index at 800 nm: {silica.compute_group_index(sample):.4f}")
print(f"GDD at 800 nm through 1 mm: "
      f"{silica.compute_group_delay_dispersion(sample, length=length).to(ureg.femtosecond**2):.2f}")
plt.show()


## Understand the result

The group index is $n_g = n - \lambda\,dn/d\lambda$, and the group
velocity is $c/n_g$. GDD is $d\tau_g/d\omega$ for a specified
propagation length. It describes how group delay changes with angular
frequency and is reported here in fs². Doubling the length doubles the GDD;
it does not change the group index.

This example uses the Malitson fused-silica Sellmeier model. The 500–1600 nm
interval stays within its source validity range and leaves room for the
finite differences used to calculate derivatives. GDD is a numerical
derivative: check convergence if you need high precision. This describes
bulk material dispersion, without waveguide dispersion.


## Explore further

Read the physics, assumptions, and suggested changes in the [full tutorial](https://martinpdes.github.io/PyOptik/docs/latest/tutorials/silica_dispersion.html).
